In [1]:
# Construye el modelo
import pyomo.environ as pe

# Resuelve el modelo
import pyomo.opt as po

In [2]:
model = pe.ConcreteModel()

### Sets

Distribuir los documentos A–G entre F1, F2 y F3 minimizando el tiempo esperado de consulta. Se sigue la formulación manuscrita: si A y B están juntos, D y E deben estar también en esa misma carpeta.

In [3]:
model.I = pe.Set(initialize=["A", "B", "C", "D", "E", "F", "G"])
model.F = pe.Set(initialize=["F1", "F2", "F3"])
model.C = pe.Set(initialize=[1, 2, 3, 4])

### Parameters

In [4]:
probability_dict = {
    1: 0.35,
    2: 0.25,
    3: 0.20,
    4: 0.20
}
model.p = pe.Param(model.C, initialize=probability_dict)

consultation_dict = {
    1: ["A", "D"],
    2: ["C", "E", "F"],
    3: ["B", "D", "G"],
    4: ["A", "B", "G"]
}
model.requested_documents = pe.Set(
    model.C, within=model.I, initialize=consultation_dict
)
model.max_documents = pe.Param(initialize=4)
model.base_time = pe.Param(initialize=1)
model.extra_time = pe.Param(initialize=0.25)

### Variables

- $x_{if}$: 1 si el documento $i$ está en la carpeta $f$.
- $\delta_f$: variable binaria auxiliar de la implicación de A y B.
- $n_{cf}$: número de documentos de la consulta $c$ almacenados en $f$.
- $y_{cf}$: 1 si la consulta $c$ necesita acceder a la carpeta $f$.
- $t_c$: tiempo relativo de la consulta $c$.

El enunciado no proporciona un tiempo absoluto; se toma como unidad el tiempo de una consulta a una sola carpeta. Los tiempos relativos para una, dos y tres carpetas son 1, 1.25 y 1.50.

In [5]:
model.x = pe.Var(model.I, model.F, within=pe.Binary)
model.delta = pe.Var(model.F, within=pe.Binary)
model.n = pe.Var(model.C, model.F, within=pe.NonNegativeIntegers)
model.y = pe.Var(model.C, model.F, within=pe.Binary)
model.t = pe.Var(model.C, within=pe.NonNegativeReals)

### Funcion objetivo

Minimizar el tiempo esperado: $\min \sum_{c\in C} p_c t_c$.

In [6]:
def obj_rule(model):
    return sum(model.p[c] * model.t[c] for c in model.C)

model.expected_time = pe.Objective(rule=obj_rule, sense=pe.minimize)

### Restricciones

In [7]:
# Cada documento se asigna exactamente a una carpeta.
def assignment_rule(model, i):
    return sum(model.x[i, f] for f in model.F) == 1

model.assignment = pe.Constraint(model.I, rule=assignment_rule)

In [8]:
# Cada carpeta contiene como máximo cuatro documentos.
def capacity_rule(model, f):
    return sum(model.x[i, f] for i in model.I) <= model.max_documents

model.capacity = pe.Constraint(model.F, rule=capacity_rule)

In [9]:
# A y F no pueden estar en la misma carpeta.
def incompatible_documents_rule(model, f):
    return model.x["A", f] + model.x["F", f] <= 1

model.incompatible_documents = pe.Constraint(
    model.F, rule=incompatible_documents_rule
)

Implicación de los apuntes, para cada $f\in F$:

$$x_{Af}+x_{Bf}\le 1+\delta_f,$$
$$x_{Df}+x_{Ef}\ge 2\delta_f.$$

Cuando A y B están juntos, la primera desigualdad obliga a $\delta_f=1$ y la segunda coloca D y E en esa carpeta.

In [10]:
def activate_delta_rule(model, f):
    return model.x["A", f] + model.x["B", f] <= 1 + model.delta[f]

model.activate_delta = pe.Constraint(model.F, rule=activate_delta_rule)

In [11]:
def same_folder_rule(model, f):
    return model.x["D", f] + model.x["E", f] >= 2 * model.delta[f]

model.same_folder = pe.Constraint(model.F, rule=same_folder_rule)

Número de documentos solicitados en cada carpeta:

$$n_{cf}=\sum_{i\in I_c}x_{if}.$$

Si $I_c$ es el conjunto de documentos de la consulta $c$, las dos desigualdades siguientes garantizan que $y_{cf}=1$ exactamente cuando $n_{cf}\ge 1$:

$$n_{cf}\le |I_c|y_{cf},\qquad y_{cf}\le n_{cf}.$$

Una carpeta puede quedar vacía: el enunciado solo establece su capacidad máxima.

In [12]:
def document_count_rule(model, c, f):
    return model.n[c, f] == sum(
        model.x[i, f] for i in model.requested_documents[c]
    )

model.document_count = pe.Constraint(
    model.C, model.F, rule=document_count_rule
)

In [13]:
def folder_required_rule(model, c, f):
    return model.n[c, f] <= len(model.requested_documents[c]) * model.y[c, f]

model.folder_required = pe.Constraint(
    model.C, model.F, rule=folder_required_rule
)

In [14]:
def folder_used_rule(model, c, f):
    return model.y[c, f] <= model.n[c, f]

model.folder_used = pe.Constraint(model.C, model.F, rule=folder_used_rule)

Latencia de cada consulta, en función del número de carpetas consultadas:

$$t_c=t_0\left[1+0.25\left(\sum_{f\in F}y_{cf}-1\right)\right],\qquad t_0=1.$$

Como cada consulta solicita al menos un documento, accede a entre una y tres carpetas. Esta igualdad reproduce los incrementos del 25 % y del 50 % del enunciado.

In [15]:
def latency_rule(model, c):
    return model.t[c] == model.base_time * (
        1 + model.extra_time * (sum(model.y[c, f] for f in model.F) - 1)
    )

model.latency = pe.Constraint(model.C, rule=latency_rule)

## Resolución con gurobi

In [16]:
solver = po.SolverFactory("gurobi_direct")

results = solver.solve(model, tee=True)

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26200.2))


CPU model: Intel(R) Core(TM) Ultra 9 185H, instruction set [SSE2|AVX|AVX2]


Thread count: 16 physical cores, 22 logical processors, using up to 22 threads


Optimize a model with 59 rows, 52 columns and 175 nonzeros (Min)


Model fingerprint: 0x8e72da54


...

Explored 1 nodes (68 simplex iterations) in 0.17 seconds (0.00 work units)


Thread count was 22 (of 22 available processors)


Solution count 4: 1.1 1.1375 1.1625 1.25 


Optimal solution found (tolerance 1.00e-04)


Best objective 1.100000000000e+00, best bound 1.100000000000e+00, gap 0.0000%


## Solución óptima

In [17]:
print("Status:", results.solver.status)
print("Termination:", results.solver.termination_condition)
print(f"TIEMPO ESPERADO MÍNIMO: {pe.value(model.expected_time):.4f}")
print(f"Incremento frente al tiempo base: {(pe.value(model.expected_time) - 1) * 100:.2f} %")

print("\nAsignación óptima:")
for f in model.F:
    documents = [i for i in model.I if pe.value(model.x[i, f]) > 0.5]
    print(f"{f}: {documents} ({len(documents)} documentos)")

print("\nConsultas:")
for c in model.C:
    folders = [f for f in model.F if pe.value(model.y[c, f]) > 0.5]
    print(
        f"C{c}: {consultation_dict[c]} -> {folders} | "
        f"carpetas: {len(folders)} | tiempo relativo: {pe.value(model.t[c]):.2f} | "
        f"probabilidad: {pe.value(model.p[c]):.0%}"
    )

Status:

ok
Termination: optimal
TIEMPO ESPERADO MÍNIMO: 1.1000
Incremento frente al tiempo base: 10.00 %

Asignación óptima:
F1: ['B', 'C', 'E', 'F'] (4 documentos)
F2: [] (0 documentos)
F3: ['A', 'D', 'G'] (3 documentos)

Consultas:
C1: ['A', 'D'] -> ['F3'] | carpetas: 1 | tiempo relativo: 1.00 | probabilidad: 35%
C2: ['C', 'E', 'F'] -> ['F1'] | carpetas: 1 | tiempo relativo: 1.00 | probabilidad: 25%
C3: ['B', 'D', 'G'] -> ['F1', 'F3'] | carpetas: 2 | tiempo relativo: 1.25 | probabilidad: 20%
C4: ['A', 'B', 'G'] -> ['F1', 'F3'] | carpetas: 2 | tiempo relativo: 1.25 | probabilidad: 20%
